# Ransomware Detection - Dissertation Visualizations
**CNN vs Transformer | University of Galway | Akhil Mudili**

Run each cell individually to generate and view each figure.

In [ ]:
import os
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# paths
BASE_DIR    = r"D:\ACS\Final Project\ransomware-detection-transformer"
RESULTS_DIR = os.path.join(BASE_DIR, "results")
VIZ_DIR     = os.path.join(BASE_DIR, "results", "visualizations")
os.makedirs(VIZ_DIR, exist_ok=True)

# confirmed results
WINDOWS       = [25, 50, 75, 100]
WINDOW_LABELS = ["25%", "50%", "75%", "100%"]

CNN_ACCURACY   = [98.14, 92.55, 95.03, 90.68]
CNN_F1         = [97.11, 89.47, 92.54, 87.20]
CNN_PRECISION  = [97.66, 86.97, 91.67, 84.37]
CNN_RECALL     = [96.58, 93.06, 93.50, 91.89]
CNN_FP         = [2, 2, 3, 2]
CNN_FN         = [1, 10, 5, 13]

TRANS_ACCURACY  = [97.52, 95.65, 95.03, 96.89]
TRANS_F1        = [96.19, 93.25, 92.70, 95.39]
TRANS_PRECISION = [96.19, 93.76, 91.09, 94.04]
TRANS_RECALL    = [96.19, 92.77, 94.63, 96.92]
TRANS_FP        = [2, 4, 2, 1]
TRANS_FN        = [2, 3, 6, 4]

CNN_CM = [
    [[31,2],[1,127]],
    [[31,2],[10,118]],
    [[30,3],[5,123]],
    [[31,2],[13,115]],
]
TRANS_CM = [
    [[31,2],[2,126]],
    [[29,4],[3,125]],
    [[31,2],[6,122]],
    [[32,1],[4,124]],
]

CNN_COLOR   = "#2196F3"
TRANS_COLOR = "#F44336"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
})

print("Setup complete. Ready to generate visualizations.")

## Figure 1 - Accuracy and F1 Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(WINDOWS))
w = 0.35

for ax, metric_cnn, metric_trans, title, ylabel in [
    (axes[0], CNN_ACCURACY, TRANS_ACCURACY, "Accuracy by Observation Window", "Accuracy (%)"),
    (axes[1], CNN_F1, TRANS_F1, "Macro F1 by Observation Window", "F1 Score (%)"),
]:
    bars1 = ax.bar(x - w/2, metric_cnn,   w, label="CNN Baseline", color=CNN_COLOR,   alpha=0.85, edgecolor="white")
    bars2 = ax.bar(x + w/2, metric_trans, w, label="Transformer",  color=TRANS_COLOR, alpha=0.85, edgecolor="white")
    for bar in bars1:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9, color=CNN_COLOR, fontweight="bold")
    for bar in bars2:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9, color=TRANS_COLOR, fontweight="bold")
    ax.set_xlabel("Observation Window"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xticks(x); ax.set_xticklabels(WINDOW_LABELS); ax.set_ylim(82, 102)
    ax.legend(framealpha=0.9); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

fig.suptitle("CNN vs Transformer - Performance Comparison Across Observation Windows", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig1_accuracy_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig1_accuracy_comparison.png")

## Figure 2 - Window Trade-off Line Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, cnn_vals, trans_vals, title, ylabel in [
    (axes[0], CNN_ACCURACY, TRANS_ACCURACY, "Accuracy vs Observation Window", "Accuracy (%)"),
    (axes[1], CNN_F1, TRANS_F1, "Macro F1 vs Observation Window", "F1 Score (%)"),
]:
    ax.plot(WINDOW_LABELS, cnn_vals,   "o-", color=CNN_COLOR,   linewidth=2.5, markersize=8, label="CNN Baseline", zorder=3)
    ax.plot(WINDOW_LABELS, trans_vals, "s-", color=TRANS_COLOR, linewidth=2.5, markersize=8, label="Transformer",  zorder=3)
    for i, (c, t) in enumerate(zip(cnn_vals, trans_vals)):
        ax.annotate(f"{c:.1f}", (WINDOW_LABELS[i], c), textcoords="offset points", xytext=(-18, 6),  fontsize=8.5, color=CNN_COLOR)
        ax.annotate(f"{t:.1f}", (WINDOW_LABELS[i], t), textcoords="offset points", xytext=(4, -14),  fontsize=8.5, color=TRANS_COLOR)
    ax.fill_between(WINDOW_LABELS, cnn_vals,   alpha=0.08, color=CNN_COLOR)
    ax.fill_between(WINDOW_LABELS, trans_vals, alpha=0.08, color=TRANS_COLOR)
    ax.set_xlabel("Observation Window Size"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_ylim(84, 101); ax.legend(framealpha=0.9)
    ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

fig.suptitle("Detection Performance vs Observation Window Size", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig2_window_tradeoff.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig2_window_tradeoff.png")

## Figure 3 - False Negatives

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(WINDOWS))
w = 0.35

bars1 = ax.bar(x - w/2, CNN_FN,   w, label="CNN Baseline", color=CNN_COLOR,   alpha=0.85, edgecolor="white")
bars2 = ax.bar(x + w/2, TRANS_FN, w, label="Transformer",  color=TRANS_COLOR, alpha=0.85, edgecolor="white")

for bar, val in zip(bars1, CNN_FN):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15, str(val),
            ha="center", va="bottom", fontsize=10, fontweight="bold", color=CNN_COLOR)
for bar, val in zip(bars2, TRANS_FN):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15, str(val),
            ha="center", va="bottom", fontsize=10, fontweight="bold", color=TRANS_COLOR)

ax.set_xlabel("Observation Window")
ax.set_ylabel("False Negatives (missed ransomware)")
ax.set_title("False Negatives per Window\n(lower is better - missed ransomware reaches the system)")
ax.set_xticks(x); ax.set_xticklabels(WINDOW_LABELS); ax.set_ylim(0, 16)
ax.legend(framealpha=0.9); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
ax.text(0.98, 0.97, "CNN misses 13 at 100% window\nTransformer misses only 4",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff9c4", edgecolor="#f0c000", alpha=0.9))

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig3_false_negatives.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig3_false_negatives.png")

## Figure 4 - Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
labels = ["Benign", "Ransomware"]

for row, (cms, model_name, color) in enumerate([
    (CNN_CM,   "CNN Baseline", CNN_COLOR),
    (TRANS_CM, "Transformer",  TRANS_COLOR),
]):
    for col, (cm, w_label) in enumerate(zip(cms, WINDOW_LABELS)):
        ax = axes[row][col]
        cm_arr = np.array(cm)
        total  = cm_arr.sum()
        im = ax.imshow(cm_arr, interpolation="nearest", cmap="Blues", vmin=0, vmax=130)
        for i in range(2):
            for j in range(2):
                val = cm_arr[i, j]
                pct = val / total * 100
                text_color = "white" if val > 65 else "black"
                ax.text(j, i, f"{val}\n({pct:.1f}%)", ha="center", va="center",
                        fontsize=9.5, color=text_color, fontweight="bold")
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(labels, fontsize=9); ax.set_yticklabels(labels, fontsize=9)
        if col == 0: ax.set_ylabel("True Label", fontsize=9)
        if row == 1: ax.set_xlabel("Predicted Label", fontsize=9)
        ax.set_title(f"{model_name}\n{w_label} Window", fontsize=10, color=color, fontweight="bold")

fig.suptitle("Confusion Matrices - CNN vs Transformer Across Observation Windows\n(Test set: 33 benign, 128 ransomware)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig4_confusion_matrices.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig4_confusion_matrices.png")

## Figure 5 - Precision and Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(WINDOWS))
w = 0.35

for ax, cnn_vals, trans_vals, title, ylabel in [
    (axes[0], CNN_PRECISION, TRANS_PRECISION, "Precision by Observation Window", "Precision (%)"),
    (axes[1], CNN_RECALL,    TRANS_RECALL,    "Recall by Observation Window",    "Recall (%)"),
]:
    ax.bar(x - w/2, cnn_vals,   w, label="CNN Baseline", color=CNN_COLOR,   alpha=0.85, edgecolor="white")
    ax.bar(x + w/2, trans_vals, w, label="Transformer",  color=TRANS_COLOR, alpha=0.85, edgecolor="white")
    for i, (c, t) in enumerate(zip(cnn_vals, trans_vals)):
        ax.text(i-w/2, c+0.2, f"{c:.1f}", ha="center", va="bottom", fontsize=8.5, color=CNN_COLOR,   fontweight="bold")
        ax.text(i+w/2, t+0.2, f"{t:.1f}", ha="center", va="bottom", fontsize=8.5, color=TRANS_COLOR, fontweight="bold")
    ax.set_xlabel("Observation Window"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xticks(x); ax.set_xticklabels(WINDOW_LABELS); ax.set_ylim(80, 102)
    ax.legend(framealpha=0.9); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

fig.suptitle("Precision and Recall - CNN vs Transformer", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig5_precision_recall.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig5_precision_recall.png")

## Figure 6 - Zero-Day Generalization

In [ ]:
# load live detection results
trans_path = os.path.join(RESULTS_DIR, "live_detection", "transformer_all_families_live.json")
cnn_path   = os.path.join(RESULTS_DIR, "live_detection", "cnn_all_families_live.json")

with open(trans_path) as f: trans_data = json.load(f)
with open(cnn_path)   as f: cnn_data   = json.load(f)

def acc_at_window(data, window):
    correct = sum(1 for r in data for w in r["windows"] if w["window"]==window and w["correct"])
    return correct / len(data) * 100 if data else 0

def family_acc_at_25(data, family):
    samples = [r for r in data if r["family"]==family]
    if not samples: return 0
    correct = sum(1 for r in samples for w in r["windows"] if w["window"]==25 and w["correct"])
    return correct / len(samples) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
windows_list = [25, 50, 75, 100]

# left: known vs zero-day
trans_known = [acc_at_window([r for r in trans_data if not r["zeroday"]], w) for w in windows_list]
trans_zd    = [acc_at_window([r for r in trans_data if r["zeroday"]], w)     for w in windows_list]
cnn_known   = [acc_at_window([r for r in cnn_data if not r["zeroday"]], w)   for w in windows_list]
cnn_zd      = [acc_at_window([r for r in cnn_data if r["zeroday"]], w)       for w in windows_list]

ax = axes[0]
x = np.arange(4); w = 0.2
ax.bar(x-1.5*w, trans_known, w, label="Transformer (Known)",    color=TRANS_COLOR, alpha=0.9)
ax.bar(x-0.5*w, trans_zd,   w, label="Transformer (Zero-day)", color=TRANS_COLOR, alpha=0.45, hatch="//")
ax.bar(x+0.5*w, cnn_known,  w, label="CNN (Known)",            color=CNN_COLOR,   alpha=0.9)
ax.bar(x+1.5*w, cnn_zd,     w, label="CNN (Zero-day)",         color=CNN_COLOR,   alpha=0.45, hatch="//")
ax.set_xticks(x); ax.set_xticklabels(["25%","50%","75%","100%"])
ax.set_xlabel("Observation Window"); ax.set_ylabel("Detection Accuracy (%)")
ax.set_title("Known vs Zero-Day Detection\nAcross Observation Windows")
ax.set_ylim(0, 115); ax.legend(fontsize=8.5, framealpha=0.9)
ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

# right: per-family at 25%
families = ["darkside","locky","ryuk","reveton","wannacry","sodinokibi","crowti","cryptodef","ctblocker"]
zd_fams  = ["wannacry","sodinokibi","crowti","cryptodef","ctblocker"]
trans_fam = [family_acc_at_25(trans_data, f) for f in families]
cnn_fam   = [family_acc_at_25(cnn_data, f)   for f in families]

ax2 = axes[1]
x2 = np.arange(len(families))
ax2.bar(x2-0.2, trans_fam, 0.35, label="Transformer", color=TRANS_COLOR, alpha=0.85)
ax2.bar(x2+0.2, cnn_fam,   0.35, label="CNN Baseline", color=CNN_COLOR,   alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels([f.capitalize() for f in families], rotation=30, ha="right", fontsize=9)
ax2.set_xlabel("Ransomware Family"); ax2.set_ylabel("Detection Accuracy at 25% Window (%)")
ax2.set_title("Per-Family Detection at 25% Window\n(red labels = zero-day families)")
ax2.set_ylim(0, 115); ax2.legend(framealpha=0.9)
ax2.yaxis.grid(True, alpha=0.3); ax2.set_axisbelow(True)
for i, label in enumerate(ax2.get_xticklabels()):
    label.set_color("#cc0000" if families[i] in zd_fams else "#333333")

fig.suptitle("Zero-Day Generalization - Live Detection Results", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig6_zeroday_generalization.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig6_zeroday_generalization.png")

## Figure 7 - Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")

col_labels = ["Window", "CNN Acc", "CNN F1", "Trans Acc", "Trans F1", "CNN FN", "Trans FN", "Winner (F1)"]
rows = []
for i, w in enumerate(WINDOW_LABELS):
    winner = "CNN" if CNN_F1[i] > TRANS_F1[i] else ("Tie" if CNN_F1[i]==TRANS_F1[i] else "Transformer")
    rows.append([w, f"{CNN_ACCURACY[i]:.2f}%", f"{CNN_F1[i]:.2f}%",
                 f"{TRANS_ACCURACY[i]:.2f}%", f"{TRANS_F1[i]:.2f}%",
                 str(CNN_FN[i]), str(TRANS_FN[i]), winner])

table = ax.table(cellText=rows, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 2.2)

for j in range(len(col_labels)):
    table[0,j].set_facecolor("#37474f"); table[0,j].set_text_props(color="white", fontweight="bold")

row_colors = ["#E3F2FD","#FFFFFF","#E3F2FD","#FFFFFF"]
for i in range(1, 5):
    for j in range(len(col_labels)):
        table[i,j].set_facecolor(row_colors[i-1])
    if rows[i-1][7] == "CNN":
        for j in [1,2]: table[i,j].set_facecolor("#C8E6C9")
    elif rows[i-1][7] == "Transformer":
        for j in [3,4]: table[i,j].set_facecolor("#FFCDD2")

ax.set_title("CNN vs Transformer - Full Results Summary", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "fig7_summary_table.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig7_summary_table.png")